# Deep dive: defect states, PT breaking, and the skin effect

**Part 2** of the tutorial series. Part 1 ([`tutorial.ipynb`](tutorial.ipynb)) built the
4N+1 Hamiltonian and ran the basic pipeline. Here we go deeper into the physics that
motivated the three model variants — and connect it to the literature:

- **Lang et al., PRB 98, 094307 (2018)** (the cSSH model; my advisor's work):
  what happens to the *mid-gap defect state* when gain/loss grows —
  1. its energy acquires an imaginary part while the real part stays pinned to zero;
  2. it can **disappear into the continuum** (localization length diverges);
  3. or undergo **ST-symmetry breaking** at an exceptional point, producing a *pair* of defect states.
- **Xu et al., Acta Phys. Sin. 72, 200301 (2023)** (my first-author review): the
  experimental landscape — classical circuits implementing these models.

We also clarify a frequent confusion: **gain/loss ≠ skin effect**. The skin effect
requires *non-reciprocal* hoppings; we demonstrate both and contrast them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ssh_lattice import build_hamiltonian, spectrum, time_evolve
%matplotlib inline

## 1. The mid-gap defect state vs gain/loss (Lang et al. 2018)

The 4N+1 geometry joins two cSSH domains at the central site $b_N$. For
$\gamma=0$ this supports a *topological mid-gap defect state* localized on $b_N$
(continuable to the Hermitian SSH defect state). Sweep $\gamma$:

- **Re $E$** stays pinned at 0 (mid-gap) — but **Im $E$** grows: the defect state
  becomes amplifying/decaying.
- The wavefunction delocalizes: its width (participation ratio) diverges —
  the state **merges into the complex continuum**.

This is exactly the fate described in Lang et al. (2018).


In [ ]:
gammas = np.linspace(0, 3.0, 13)
N, J1, J2 = 10, 0.5, 1.5
bN = 2*N
data = []
for g in gammas:
    H = build_hamiltonian(N, g, J1, J2, 'a')
    ev, vec = spectrum(H)
    idx = np.argmin(np.abs(ev.real))          # the mid-gap state
    prob = np.abs(vec[:, idx])**2; prob /= prob.sum()
    data.append(dict(g=g, E=ev[idx], prob_bN=prob[bN],
                     width=1.0/np.sum(prob**2)))
Re  = [d['E'].real for d in data]
Im  = [d['E'].imag for d in data]
width = [d['width'] for d in data]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(gammas, Re, 'o-'); axes[0].set_xlabel(r'$\gamma$'); axes[0].set_ylabel(r'Re $E$')
axes[0].set_title('Real part pinned to mid-gap'); axes[0].grid(alpha=0.3)
axes[1].plot(gammas, Im, 'o-', color='tab:red'); axes[1].set_xlabel(r'$\gamma$'); axes[1].set_ylabel(r'Im $E$')
axes[1].set_title('Imaginary part grows: amplification'); axes[1].grid(alpha=0.3)
axes[2].plot(gammas, width, 'o-', color='tab:green'); axes[2].set_xlabel(r'$\gamma$'); axes[2].set_ylabel('width (sites)')
axes[2].set_title('Localization length diverges: into the continuum'); axes[2].grid(alpha=0.3)
fig.suptitle('Mid-gap defect state under gain/loss (variant a, N=10)')
plt.tight_layout(); plt.show()

**Read-out:** the real part stays exactly at 0 (blue), the imaginary part grows
(red) — the defect state turns into an amplifying mode — and the wavefunction width
diverges (green): the state merges into the bulk continuum. Both effects require the
bulk to have undergone PT symmetry breaking, exactly as Lang et al. describe.

Check it: at $\gamma=2$ the "defect" is spread over the whole chain — try plotting
the wavefunction at $\gamma=0$ vs $\gamma=2$ below.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, g in zip(axes, [0.0, 2.0]):
    H = build_hamiltonian(N, g, J1, J2, 'a')
    ev, vec = spectrum(H)
    idx = np.argmin(np.abs(ev.real))
    prob = np.abs(vec[:, idx])**2; prob /= prob.sum()
    ax.bar(np.arange(1, 4*N+2), prob, width=1)
    ax.axvline(bN+1, color='r', ls='--', lw=1)
    ax.set_title(rf'$\gamma$={g}: $E={ev[idx]:.3f}$, width={1/np.sum(prob**2):.1f}')
    ax.set_xlabel('Lattice site'); ax.set_ylabel('probability'); ax.grid(alpha=0.2)
fig.suptitle('Mid-gap defect state: localized at b_N (left) vs dissolved into the continuum (right)')
plt.tight_layout(); plt.show()

## 2. Non-unitary long-time evolution (variant c)

Variant `c` keeps gain on $b_N$; with $\gamma=2.5$ the spectrum has eigenvalues with
$\mathrm{Im}\,E > 0$, so the time evolution $e^{-iHt}$ is **non-unitary and the norm
grows exponentially** — there is no probability conservation in an open system.

Two ways to look at it:

1. **Raw evolution**: $\|\psi(t)\| \sim e^{\gamma_{\max} t}$ with
   $\gamma_{\max}=\max \mathrm{Im}\,E$ — the system is an amplifier.
2. **Normalized evolution** (common in the literature): the long-time state converges
   to the eigenstate with the largest imaginary energy — the dominant amplifying mode.


In [ ]:
p = dict(N=20, gamma=2.5, J1=0.5, J2=1.5, t_max=10.0, dt=0.02)   # short enough to stay finite
Hc = build_hamiltonian(p['N'], p['gamma'], p['J1'], p['J2'], 'c')
evc, _ = spectrum(Hc)
gmax = np.abs(evc.imag).max()
print(f'max |Im E| = {gmax:.3f}  ->  predicted norm growth rate e^({gmax:.2f}t)')

psi0 = np.zeros(4*p['N']+1, dtype=complex); psi0[2*p['N']] = 1.0
t, pt, _, _ = time_evolve(Hc, psi0, p['t_max'], p['dt'], 2*p['N'])
norms = np.linalg.norm(pt, axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.semilogy(t, norms, lw=2)
ax1.set_xlabel('Time'); ax1.set_ylabel(r'$\|\psi(t)\|$ (log)')
ax1.set_title('Non-unitary evolution: exponential norm growth'); ax1.grid(alpha=0.3)

# normalized evolution -> dominant amplifying mode
ptn = pt / np.linalg.norm(pt, axis=0, keepdims=True)
fp = np.abs(ptn[:, -1])**2
ax2.bar(np.arange(1, len(fp)+1), fp, width=1)
ax2.set_xlabel('Lattice site'); ax2.set_ylabel('probability (normalized)')
ax2.set_title('Normalized final state = dominant amplifying eigenmode'); ax2.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## 3. Skin effect vs gain/loss — two different kinds of non-Hermiticity

A common confusion: **gain/loss (on-site $\pm i\gamma$) does NOT cause the skin
effect**. The non-Hermitian skin effect (all eigenstates piled at one edge) requires
**non-reciprocal hoppings** ($H_{ij} \neq H_{ji}$).

Both are reviewed in my survey (Xu et al. 2023): circuits implement gain/loss with
negative-resistance elements (INIC) and non-reciprocity with active converters.

Demonstration below: a non-reciprocal SSH chain (hoppings $t\pm\delta$ in one
direction, $t\mp\delta$ in the other) piles **every** eigenstate at the right edge —
contrast with the cSSH chain where no such bulk pile-up occurs.


In [ ]:
# non-reciprocal SSH chain
Nr = 30
Hnr = np.zeros((2*Nr, 2*Nr), dtype=complex)
for i in range(Nr):
    Hnr[2*i, 2*i+1] = 0.9; Hnr[2*i+1, 2*i] = 1.1
for i in range(Nr-1):
    Hnr[2*i+1, 2*i+2] = 0.9; Hnr[2*i+2, 2*i+1] = 1.1
evnr, vecnr = spectrum(Hnr)
pos = np.arange(1, 2*Nr+1)
meanpos = np.array([pos @ (np.abs(vecnr[:,i])**2) / np.sum(np.abs(vecnr[:,i])**2)
                    for i in range(2*Nr)])
# total density of all eigenstates
density = np.sum(np.abs(vecnr)**2, axis=1)
density /= density.max()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(np.arange(1, 2*Nr+1), density, width=1)
axes[0].set_xlabel('Lattice site'); axes[0].set_ylabel('total eigenstate density')
axes[0].set_title('Non-reciprocal SSH: skin effect (all states at one edge)')
axes[0].axvline(2*Nr, color='r', ls='--', lw=1); axes[0].grid(alpha=0.2)

# cSSH for contrast: same total density
Hc2 = build_hamiltonian(15, 1.5, 0.5, 1.5, 'a')
evc2, vec2 = spectrum(Hc2)
density2 = np.sum(np.abs(vec2)**2, axis=1); density2 /= density2.max()
axes[1].bar(np.arange(1, len(density2)+1), density2, width=1)
axes[1].set_xlabel('Lattice site'); axes[1].set_ylabel('total eigenstate density')
axes[1].set_title('cSSH (gain/loss only): no skin effect')
axes[1].grid(alpha=0.2)
fig.suptitle('Non-reciprocity piles states at the boundary; on-site gain/loss does not')
plt.tight_layout(); plt.show()

## Summary

| Phenomenon | What causes it | In this repo |
|------------|---------------|--------------|
| Mid-gap defect state (real part pinned to 0) | topology + sublattice symmetry | variant a/b/c at $\gamma=0$ |
| Amplification (Im $E \neq 0$) | on-site gain/loss | any variant, $\gamma>0$ |
| Disappearance into continuum | localization length divergence under strong $\gamma$ | variant a, $\gamma\gtrsim 1.5$ |
| Skin effect | **non-reciprocal** hoppings | extra demo above (not a 4N+1 variant) |

All of these are discussed in the context of classical-circuit emulation in my
review (Xu et al. 2023); the cSSH defect-state theory is from Lang et al. (PRB 2018).
